# AI Voice Cloning — CosyVoice3 Fine-Tuning Engineering

Lei Wang

Five short notes from a CosyVoice3 LLM fine-tune. The model and the training algorithm are [FunAudioLLM/CosyVoice](https://github.com/FunAudioLLM/CosyVoice), Apache-2.0. See `THIRD_PARTY_NOTICES.md`.

This notebook is not a voice-cloning product and not a training pipeline. The code cells only check configuration values. They do not download weights, read a recording, or start `train.py`.


## 1. Training Environment and Model Setup

**Problem.** A fine-tune has to stay on one known copy of CosyVoice, and it should move only the LLM. Flow and HiFi-GAN stay at the released weights.

**Snippet.** Selected from the training template. This cell prints the pin. It does not clone the repository.


In [1]:
COSYVOICE_COMMIT = "074ca6dc9e80a2f424f1f74b48bdd7d3fea531cc"
MODEL_ID = "FunAudioLLM/Fun-CosyVoice3-0.5B-2512"
TRAINED_MODULE = "llm"
CHECKPOINT = "llm.pt"

print(MODEL_ID)
print(TRAINED_MODULE, CHECKPOINT)
print(COSYVOICE_COMMIT)


FunAudioLLM/Fun-CosyVoice3-0.5B-2512
llm llm.pt
074ca6dc9e80a2f424f1f74b48bdd7d3fea531cc


**Solution.** Training continues from the official `llm.pt` at that commit. The other released networks are not optimized.

**Limit.** The commit id does not install CosyVoice. The weights are not in this repository.


## 2. Training Epoch Configuration

**Problem.** Upstream `train.py` has no epoch argument. The loop is `range(start_epoch + 1, max_epoch)` inside the official yaml.

**Snippet.** Adapted from `formal_config_text`. The numbers below are an example of the formula, not a logged training run.


In [2]:
def formal_max_epoch(start_epoch, epochs):
    if type(epochs) is not int or epochs <= 0:
        raise ValueError("epochs must be a positive int")
    if type(start_epoch) is not int or start_epoch < 0:
        raise ValueError("start_epoch must be a non-negative int")
    return start_epoch + 1 + epochs

print("one epoch from the official start:", formal_max_epoch(0, 1))
print("learning rate stays the official 1e-5 with constantlr")


one epoch from the official start: 2
learning rate stays the official 1e-5 with constantlr


**Solution.** `max_epoch` is written into a copy of the official yaml. The official file is left unchanged. Learning rate stays `1e-5` with `constantlr`.

**Limit.** `formal_max_epoch(0, 1)` returns `2`, which means one training epoch when the start epoch is 0. It does not run a trainer.


## 3. Smoke Test and Formal Training Safety Controls

**Problem.** A full run should not start because a notebook was opened or because a placeholder was filled in.

**Snippet.** Adapted from `formal_lock`. Smoke in the real run was one epoch on 8 clips. This cell only returns true or false.


In [3]:
def formal_training_allowed(smoke_passed, epochs, confirmed):
    if not smoke_passed or type(epochs) is not int or epochs <= 0 or confirmed is not True:
        return False
    return True

print("smoke pass, epochs filled, confirm still false:", formal_training_allowed(True, 1, False))
print("smoke pass, epochs filled, confirm true:", formal_training_allowed(True, 1, True))


smoke pass, epochs filled, confirm still false: False
smoke pass, epochs filled, confirm true: True


**Solution.** Formal training needs three separate facts: a smoke pass, a positive integer epoch count, and `confirmed is True`. Setting the epoch count alone stays locked.

**Limit.** Returning true here does not start training. The real lock also checks that the smoke checkpoint file is still the one that passed.


## 4. GPU Memory Optimization on NVIDIA T4 16GB

**Problem.** The official formal batch, `max_frames_in_batch` 2000 with `accum_grad` 2, did not fit the first formal batch on a 16 GB T4.

**Snippet.** Adapted from the template constants. This compares the nominal frame product. It does not allocate a GPU batch.


In [4]:
OFFICIAL_MAX_FRAMES = 2000
OFFICIAL_ACCUM = 2
T4_MAX_FRAMES = 1000
T4_ACCUM = 4

def accumulated_frames(max_frames, accum_grad):
    return max_frames * accum_grad

official = accumulated_frames(OFFICIAL_MAX_FRAMES, OFFICIAL_ACCUM)
adapted = accumulated_frames(T4_MAX_FRAMES, T4_ACCUM)
print("official product", official)
print("T4 product", adapted)
print("same nominal product", official == adapted)


official product 4000
T4 product 4000
same nominal product True


**Solution.** The formal run used 1000 frames and an accumulation of 4. Each optimizer step still targets the same nominal frame count, 4000, while the per-step batch is smaller. Learning rate was not changed.

**Limit.** Equal products do not prove identical loss or identical updates. No loss curve is shown here. Smoke training kept the official 2000-frame setting and used `accum_grad` 1, because that run is only 8 clips and one epoch.


## 5. Training Execution and Checkpoint Validation

**Problem.** The trainer writes `init.pt` before any epoch. A finished run has to be told apart from that file.

**Snippet.** Adapted from the checkpoint gate. The dictionary below is synthetic. It is not a measured result.


In [5]:
def checkpoint_record_ok(record):
    if record.get("step", 0) <= 0:
        return False, "step must be > 0"
    if record.get("changed_tensors", 0) <= 0:
        return False, "parameters did not change"
    if record.get("nonfinite") is not False:
        return False, "non-finite values"
    if record.get("shape_match") is not True:
        return False, "tensor names or shapes differ"
    if record.get("extra_epoch") is not False:
        return False, "checkpoint beyond the requested epoch"
    return True, "record passed these checks"

synthetic_pass = {
    "step": 1,
    "changed_tensors": 1,
    "nonfinite": False,
    "shape_match": True,
    "extra_epoch": False,
}
synthetic_init = dict(synthetic_pass, step=0, changed_tensors=0)
print("synthetic pass", checkpoint_record_ok(synthetic_pass))
print("synthetic init", checkpoint_record_ok(synthetic_init))


synthetic pass (True, 'record passed these checks')
synthetic init (False, 'step must be > 0')


**Solution.** A real pass also loads the epoch checkpoint, compares it with `init.pt`, and reads the training log. `init.pt` is only the comparison baseline.

**Limit.** This cell does not load a file and does not start training. The real run used the upstream stage-5 entry: one GPU, fp32, `torch_ddp`, LLM only. That command is not in this notebook.
